<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-07-fine-tuning/lesson-7.2-data-preparation/notebooks/exercises-7.2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7.2 — Data Preparation for Fine-Tuning
Netsetos GenAI Course v2.0 | Module 7

Data formats, chat templates, quality filtering, annotation tools, versioning, India data prep


In [ ]:
# pip install transformers datasets datasketch presidio-analyzer presidio-anonymizer -q
print('Ready for Data Preparation')


## Ex 1: Format Conversion (Alpaca → OpenAI)


In [ ]:
def alpaca_to_openai(example, system_prompt=''):
    messages = []
    if system_prompt:
        messages.append({'role':'system','content':system_prompt})
    user = example['instruction']
    if example.get('input','').strip():
        user += '\n\n' + example['input']
    messages.append({'role':'user','content':user})
    messages.append({'role':'assistant','content':example['output']})
    return {'messages': messages}

alpaca = {'instruction':'Classify sentiment','input':'Great product!','output':'Positive'}
result = alpaca_to_openai(alpaca, 'You are a sentiment classifier.')
for m in result['messages']:
    print(f"  {m['role']:10s}: {m['content']}")


## Ex 2: ShareGPT → OpenAI Conversion


In [ ]:
ROLE_MAP = {'human':'user','gpt':'assistant','system':'system'}

def sharegpt_to_openai(example):
    messages = []
    if example.get('system'):
        messages.append({'role':'system','content':example['system']})
    for turn in example.get('conversations',[]):
        role = ROLE_MAP.get(turn.get('from',''), turn.get('from',''))
        messages.append({'role':role,'content':turn.get('value','')})
    return {'messages': messages}

sgpt = {'conversations':[
    {'from':'human','value':'Plan a trip to Japan'},
    {'from':'gpt','value':'Here are key things to know...'},
    {'from':'human','value':'How long for Tokyo?'},
    {'from':'gpt','value':'I recommend 10-14 days...'}
]}
result = sharegpt_to_openai(sgpt)
for m in result['messages']:
    print(f"  {m['role']:10s}: {m['content'][:50]}...")


## Ex 3: Chat Template Comparison


In [ ]:
# from transformers import AutoTokenizer
# models = ['meta-llama/Llama-3.1-8B-Instruct',
#           'Qwen/Qwen2.5-7B-Instruct',
#           'google/gemma-2-2b-it']
# messages = [{'role':'user','content':'Hello!'}]
# for m in models:
#     tok = AutoTokenizer.from_pretrained(m)
#     formatted = tok.apply_chat_template(messages, tokenize=False)
#     print(f'{m}:\n{formatted}\n')

print('Chat template differences:')
for model, tokens in [
    ('Llama 3','<|start_header_id|>user<|end_header_id|>\nHello!<|eot_id|>'),
    ('Mistral','[INST] Hello! [/INST]'),
    ('Qwen 2.5','<|im_start|>user\nHello!<|im_end|>'),
    ('Gemma','<start_of_turn>user\nHello!<end_of_turn>'),
]: print(f'  {model:12s}: {tokens}')


## Ex 4: Special Tokens Check


In [ ]:
print('Special tokens checklist before fine-tuning:')
print()
print('# Check special tokens')
print('print(f"BOS: {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")')
print('print(f"EOS: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")')
print('print(f"PAD: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")')
print()
print('# NEVER do this:')
print('# tokenizer.pad_token = tokenizer.eos_token  # CAUSES BUGS')
print()
print('# ALWAYS do this:')
print('tokenizer.add_special_tokens({"pad_token": "[PAD]"})')
print('model.resize_token_embeddings(len(tokenizer))')
print('tokenizer.padding_side = "right"  # training')
print('tokenizer.padding_side = "left"   # inference')


## Ex 5: AlpaGasus Quality Scoring


In [ ]:
def score_example(item):
    '''Simulate GPT-4o-mini quality scoring'''
    score = 3.0  # baseline
    if len(item.get('output','')) > 100: score += 0.5
    if '?' not in item.get('output',''): score += 0.3  # answers, not questions
    if len(item.get('instruction','')) > 20: score += 0.5
    if item.get('has_example', False): score += 0.7
    return min(score, 5.0)

examples = [
    {'instruction':'Hi','output':'Hello!'},  # low quality
    {'instruction':'Explain the difference between LoRA and QLoRA for LLM fine-tuning',
     'output':'LoRA freezes base weights and trains low-rank adapters (0.1-5% of params)...',
     'has_example':True},  # high quality
]
for i, ex in enumerate(examples):
    s = score_example(ex)
    keep = '✅ KEEP' if s >= 4.5 else '❌ FILTER'
    print(f'  Example {i+1}: score={s:.1f} → {keep}')


## Ex 6: MinHash Deduplication


In [ ]:
# from datasketch import MinHash, MinHashLSH

def simple_dedup(documents, threshold=0.7):
    '''Simple Jaccard-based dedup (production: use MinHash/LSH)'''
    seen = []
    unique = []
    for doc in documents:
        words = set(doc.lower().split())
        is_dup = False
        for s in seen:
            jaccard = len(words & s) / len(words | s) if words | s else 0
            if jaccard > threshold:
                is_dup = True; break
        if not is_dup:
            seen.append(words)
            unique.append(doc)
    return unique

docs = [
    'How to fine-tune a language model with LoRA',
    'Fine-tuning language models using LoRA technique',  # near-dup
    'What is the capital of France',  # unique
    'The capital of France is Paris',  # different topic
]
result = simple_dedup(docs)
print(f'Input: {len(docs)} → Output: {len(result)} (removed {len(docs)-len(result)} duplicates)')
for r in result: print(f'  {r}')


## Ex 7: PII Removal


In [ ]:
import re

def remove_pii_simple(text):
    # Aadhaar: XXXX XXXX XXXX
    text = re.sub(r'\b\d{4}\s\d{4}\s\d{4}\b', '<AADHAAR>', text)
    # PAN: ABCDE1234F
    text = re.sub(r'\b[A-Z]{5}[0-9]{4}[A-Z]\b', '<PAN>', text)
    # Email
    text = re.sub(r'\b[\w.-]+@[\w.-]+\.\w+\b', '<EMAIL>', text)
    # Phone (Indian)
    text = re.sub(r'\b(?:\+91|0)?[789]\d{9}\b', '<PHONE>', text)
    return text

test = 'Customer Rahul (rahul@email.com, Aadhaar: 1234 5678 9012, PAN: ABCDE1234F) called +919876543210'
print('Before:', test)
print('After: ', remove_pii_simple(test))


## Ex 8: Tokenizer Fertility Benchmark


In [ ]:
print('Tokenizer fertility comparison (tokens per Hindi word):')
print()
hindi = 'भारत में कृत्रिम बुद्धिमत्ता का विकास तेज़ी से हो रहा है'
print(f'Hindi text: {hindi}')
print(f'Word count: {len(hindi.split())}')
print()
for model, fertility in [
    ('Llama 3.1 8B', '4-8 tokens/word (inefficient)'),
    ('Gemma 2', '~2.2 tokens/word (good)'),
    ('Qwen 2.5', '~3-5 tokens/word (moderate)'),
    ('Sarvam-1', '1.4-2.1 tokens/word (best for Indic)'),
]: print(f'  {model:15s}: {fertility}')
print()
print('Impact at 4096 max_length:')
print('  English (~1.3): ~3000 words fit')
print('  Hindi Llama (~6): ~680 words fit (4.4× less!)')
print('  Hindi Sarvam (~1.8): ~2275 words fit')


---
## Recap
Fine-tuning data preparation is 80% of the work. Use OpenAI messages JSONL format (ChatML) as the default. ALWAYS apply chat templates with tokenizer.apply_chat_template() — wrong templates are the #1 silent failure. Never set pad_token = eos_token. Quality filtering (AlpaGasus): score with GPT-4o-mini, keep top 15-25% — 9K filtered > 52K raw. Pipeline order: length → language → dedup (MinHash/LSH) → toxicity → PII (Presidio with placeholders) → contamination check → LLM scoring. Tools: Distilabel generates → Lilac explores → Argilla annotates → TRL trains. Split 80/10/10 stratified, min 50-100 validation. Version with DVC + HF Hub. India: AI4Bharat Sangraha (251B tokens), normalize Unicode NFC first, measure tokenizer fertility (Sarvam-1: 1.4-2.1 vs Llama: 4-8 for Hindi), DPDPA compliance (consent-centric, ₹250 crore penalties).
